# 04b — Video Mode 2 (low-VRAM alternate): Wan VACE 1.3B "swap anything"

The cheaper Mode 2 path (ComfyUI workflow `04_video_mode2_vace.json`, replaced here). Use it when:
- you're on a **16–24 GB card** (L4/4090) — 1.3B at 480p needs ~8–10 GB
- the source clip has **multi-subject** scenes and you need a precise SAM2 mask (04a's built-in
  mask extractor is single-person only)

**Pipeline** (`WanVACEPipeline`, verified against diffusers source):
1. **SAM 2** tracks the source subject → binary mask video (white = regenerate, black = preserve —
   the VACE convention: *black regions are conditioning/preserved, white regions are generated*).
2. `WanVACEPipeline(prompt, video=source frames, mask=mask frames, reference_images=[your char])`
   regenerates the masked region as your character, keeping the background.

**vs 04a:** VACE is a general editing model (mask + reference), not a dedicated animate model —
expression/pose replication is weaker than Wan2.2-Animate, but it's 3–4× lighter and gives you
explicit mask control.

**Inputs:** source video (`.mp4`), a character still (02a output / training ref).

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
WIDTH, HEIGHT  = 832, 480    # 1.3B is 480p-class; 14B variant can do 720p (§8)
MAX_FRAMES     = 81          # 4k+1; longer clips → more VRAM + time
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
VID_OUT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/mode2_vace'
os.makedirs(VID_OUT, exist_ok=True)
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

SOURCE_VIDEO  = None   # set via upload below or a Drive path
CHARACTER_REF = None
print(f'Video out: {VID_OUT}')

## 2. Upload inputs

In [ ]:
from google.colab import files
print('Upload the SOURCE VIDEO (.mp4):')
SOURCE_VIDEO = list(files.upload())[0]
print('Upload the CHARACTER REFERENCE still:')
CHARACTER_REF = list(files.upload())[0]
print('SOURCE_VIDEO  =', SOURCE_VIDEO)
print('CHARACTER_REF =', CHARACTER_REF)

## 3. Install (diffusers for VACE + SAM2 for masking)
SAM2 is the **Meta repo** (`facebookresearch/sam2`) cloned + `pip install -e .` (the canonical
video-predictor build; the bare `sam2` pip wheel is a different thing). Checkpoint from
`facebook/sam2`, config from the repo root. Everything runs in the system env — no venv needed.

In [ ]:
# diffusers for the VACE pipeline (system env)
!pip install -q uv
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub ftfy

# SAM2 (Meta) for subject masking — clone + editable install (canonical; the pip
# "sam2" wheel is not the Meta video-predictor build). Reuses Colab's torch.
import os, subprocess
SAM2_REPO = '/content/sam2'
if not os.path.exists(SAM2_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/sam2.git {SAM2_REPO}
!pip install -e {SAM2_REPO}

# SAM2.1 large checkpoint (~1.5 GB) into the HF cache on Drive
from huggingface_hub import hf_hub_download
SAM2_CKPT = hf_hub_download('facebook/sam2', 'sam2.1_hiera_large.pt')
print('SAM2 checkpoint →', SAM2_CKPT)

# Video-predictor config lives at the REPO root (configs/sam2.1/...), not inside the package
SAM2_CFG = f'{SAM2_REPO}/configs/sam2.1/sam2.1_hiera_l.yaml'
print('SAM2 config →', SAM2_CFG, '(exists:', os.path.exists(SAM2_CFG), ')')

import torch, diffusers
from sam2.build_sam import build_sam2_video_predictor   # confirms the import path
print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| sam2 OK')
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 4. SAM 2 subject tracking → mask video
Pick **one point** on the subject in the first frame (normalized coords, 0-1) and SAM2 tracks it
across all frames. Review the mask frames before the expensive VACE run — a drifting mask = a
drifting regeneration region.

If auto point-selection is off, tweak `PT` (or use the box variant noted below).

In [ ]:
import numpy as np, time, os, torch
from PIL import Image
from diffusers.utils import load_video
from sam2.build_sam import build_sam2_video_predictor

# Load + trim the source to MAX_FRAMES (4k+1) and target resolution
frames = load_video(SOURCE_VIDEO)[:MAX_FRAMES]
frames = [f.convert('RGB').resize((WIDTH, HEIGHT)) for f in frames]
n = len(frames)
print(f'{n} frames @ {WIDTH}x{HEIGHT}')

predictor = build_sam2_video_predictor(SAM2_CFG, SAM2_CKPT)
video_np = np.stack([np.asarray(f) for f in frames])  # HxWxC uint8
with torch.inference_mode():
    state = predictor.init_state(video_np)

    # ── prompt: a single foreground point on the subject in frame 0 ──────────
    PT = (0.5, 0.4)     # (x, y) normalized — adjust if your subject is elsewhere
    W_, H_ = WIDTH, HEIGHT
    frame_idx, object_ids, mask0 = predictor.add_new_points_or_box(
        state, frame_idx=0,
        points=[(int(PT[0]*W_), int(PT[1]*H_))],
        labels=[1],
    )
    # (alternative: predictor.add_new_boxes(state, frame_idx=0, boxes=[(x0,y0,x1,y1)]))

    masks = [mask0[0]]   # first frame mask (HxW bool), index [0] = object 0
    for i in range(1, n):
        _, _, m = predictor.propagate_in_video(state, frame_idx=i)
        masks.append(m[0])
        if i % 20 == 0:
            print(f'  tracked {i}/{n}')
del state, predictor
torch.cuda.empty_cache()

mask_pils = [Image.fromarray((m * 255).astype('uint8')) for m in masks]

# quick review montage (white = region that will be REGENERATED)
from IPython.display import display, HTML
import base64
def b64p(im):
    im = im.convert('L'); im.thumbnail((256, 256))
    from io import BytesIO
    b = BytesIO(); im.save(b, 'JPEG', quality=80)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}" style="border:1px solid #555">'
sample = [0, n//4, n//2, 3*n//4, n-1]
display(HTML('<h4>Mask (white = regenerate)</h4><div style="display:flex;gap:6px">' +
             ''.join(b64p(mask_pils[i]) for i in sample) + '</div>'))
print('If the mask misses the subject, change PT in the cell above and rerun.')

## 5. Load `WanVACEPipeline` (1.3B) and run swap-anything
1.3B at 480p runs in ~8–10 GB → fits T4/L4. First run downloads ~7 GB.

In [ ]:
import torch, logging
from diffusers import AutoencoderKLWan, WanVACEPipeline
from diffusers.utils import export_to_video, load_image
from pathlib import Path

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU ~{vram_gb:.0f} GB')
LOG = '/content/vace_load.log'
logging.basicConfig(filename=LOG, level=logging.INFO)

model_id = "Wan-AI/Wan2.1-VACE-1.3B-diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(model_id, vae=vae, dtype=torch.bfloat16)

if vram_gb >= 30:
    pipe.to('cuda'); print('Strategy: resident')
else:
    from diffusers.hooks import apply_group_offloading
    onload, offload = torch.device('cuda'), torch.device('cpu')
    apply_group_offloading(pipe.text_encoder, onload_device=onload, offload_device=offload,
                           offload_type='block_level', num_blocks_per_group=4)
    pipe.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                          offload_type='leaf_level', use_stream=True)
    print('Strategy: group offloading')
print('✅ WanVACEPipeline ready. Log →', LOG)

In [ ]:
# ── Swap-anything call ─────────────────────────────────────────────────────
# VACE mask convention: BLACK = condition/preserve, WHITE = generate.
char_ref = load_image(CHARACTER_REF).convert('RGB').resize((WIDTH, HEIGHT))

prompt   = (f'{TRIGGER_TOKEN}, the person in the scene, consistent character identity, '
            'natural motion, seamless integration into the environment')
negative = 'blurry, low quality, deformed, extra limbs, inconsistent identity, watermark'

g = torch.Generator(device='cuda').manual_seed(0)
out = pipe(
    prompt=prompt,
    negative_prompt=negative,
    video=frames,                 # source frames (conditioning for black-mask areas)
    mask=mask_pils,               # per-frame masks (white = regenerate)
    reference_images=[char_ref],  # who to put in the white regions
    conditioning_scale=1.0,
    height=HEIGHT, width=WIDTH,
    num_frames=len(frames),
    num_inference_steps=50,
    guidance_scale=5.0,
    generator=g,
).frames[0]

ts = time.strftime('%Y%m%d_%H%M%S')
out_path = Path(VID_OUT) / f'{ts}_swap.mp4'
export_to_video(out, str(out_path), fps=16)
print(f'✅ swapped video → {out_path}')

from IPython.display import Video, display
display(Video(str(out_path), width=720))

## 6. Log to metadata

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}
clips = sorted(glob.glob(f'{VID_OUT}/*.mp4'))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode2_vace',
    'model': 'Wan2.1-VACE-1.3B (swap-anything)',
    'clips': clips[-5:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(clips)} clips in {VID_OUT}')

## 7. Commented alternates (try later)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) VACE 14B @ 720p — same swap-anything, much higher fidelity. ~16-24 GB.
#    Change model_id to "Wan-AI/Wan2.1-VACE-14B-diffusers", WIDTH=1280, HEIGHT=720.
# ─────────────────────────────────────────────────────────────────────────
# model_id = "Wan-AI/Wan2.1-VACE-14B-diffusers"
# ... (reload vae + pipe from cell 5 with the new id and bigger dims)

# ─────────────────────────────────────────────────────────────────────────
# B) DWPose control-to-video — drive generation from a skeleton video instead
#    of a pixel mask: draw per-frame DWPose keypoints, pass that as `video` with
#    an ALL-BLACK mask (pure conditioning) + reference_images. Gives strong pose
#    match, looser background fidelity. DWPose (Apache-2.0, NOT OpenPose) via the
#    dw-ll onnx model + onnxruntime; extraction is the same loop as 04a's pose step.
# ─────────────────────────────────────────────────────────────────────────
# skeleton_frames = [draw_dwpose(f) for f in frames]          # per-frame keypoint draw
# black_masks = [Image.new('L', (WIDTH, HEIGHT), 0)] * len(frames)   # all black = condition
# out = pipe(prompt=..., video=skeleton_frames, mask=black_masks,
#            reference_images=[char_ref], conditioning_scale=1.0,
#            height=HEIGHT, width=WIDTH, num_frames=len(frames)).frames[0]

# ─────────────────────────────────────────────────────────────────────────
# C) First/last-frame conditioning via VACE — pass start (+optional last) stills
#    as `video` with matching masks (white where you want generation) to steer
#    the clip's endpoints; overlaps with 03a/03b but in the VACE family.
# ─────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────
# D) conditioning_scale knob — lower it (0.5-0.8) if the background bleeds the
#    regenerated subject, or the source pose dominates too hard.
# ─────────────────────────────────────────────────────────────────────────

print('Section 7: alternates commented out.')